# Aula 12 — Breakpoints: pausar para aprovação e retomar

**Objetivo:** o primeiro humano-no-loop de verdade. Ao final você deve saber
pausar antes das tools (`interrupt_before`), inspecionar o pedido pendente com
`get_state` e retomar com `stream(None, config)`.

**Pré-requisitos:** Aula 06 (checkpointer + thread) e Aula 11 (stream).

**Como usar:** rode em ordem. Autocontido (1 arquivo no Colab). Comece pela
CÉLULA 0. Cole sua chave na CÉLULA 1 (só precisa no Colab/sem .env).
Com `GROQ_API_KEY` usa Groq real; sem chave, Fake offline.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph langchain-groq python-dotenv
print('dependências prontas.')

In [ ]:
# CÉLULA 1 — chave do Groq (opcional: sem ela, tudo roda no Fake offline)
import getpass
import os
from dotenv import load_dotenv

load_dotenv()  # lê .env se existir (uso local); no Colab não há .env

if not os.getenv('GROQ_API_KEY'):
    try:
        os.environ['GROQ_API_KEY'] = getpass.getpass('GROQ_API_KEY (digitação oculta): ')
    except Exception:
        print('sem entrada interativa -> seguirei em modo Fake offline.')

print('chave presente:', bool(os.getenv('GROQ_API_KEY')))
# NUNCA dê print na chave. Verificamos só presença (True/False).

## 1. Analogia: o fiscal que carimba antes do caixa

Na lanchonete, contas acima de R$ 50 passam pelo fiscal: ele **pausa** o pedido,
**lê** o que o caixa vai executar, **carimba** (aprova) e só então o caixa roda.

| Lanchonete | No LangGraph | O que é de verdade |
|---|---|---|
| fiscal na porta do caixa | **`interrupt_before=['tools']`** | pausa antes do nó, todo `invoke`/`stream` |
| pedido parado na bancada | **`__interrupt__`** | chunk avisando que pausou (não é nó seu) |
| 'próximo: caixa' | **`get_state(cfg).next`** | tupla com o próximo nó (`('tools',)`) |
| carimbo 'pode passar' | **`stream(None, cfg)`** | retoma do checkpoint (input `None` = 'continua') |
| comanda guardada | **checkpointer** | sem sessão salva, não há de onde retomar |

Sem fiscal (Aulas 04–11), a tool executa sozinha. Com fiscal, nada executa sem
passar pela pausa — é assim que dinheiro de verdade (e dados sensíveis) se protegem.

In [ ]:
# CÉLULA 2 — fábrica do agente com fiscal (LLM fresco por grafo)
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import MessagesState, StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

SISTEMA = 'Você é um assistente que faz contas. Responda em português, de forma curta.'
MODELO = 'openai/gpt-oss-20b'

def add(a: int, b: int) -> int:
    """Soma a e b."""
    return a + b

TOOLS = [add]

def get_llm():
    if os.getenv('GROQ_API_KEY'):
        try:
            from langchain_groq import ChatGroq
        except ImportError:
            print('AVISO: langchain_groq ausente -> Fake offline.')
        else:
            return ChatGroq(model=MODELO, temperature=0,
                            reasoning_effort='low').bind_tools(TOOLS)
    from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
    print('AVISO: modo Fake offline (2 falas gravadas).')
    return FakeMessagesListChatModel(responses=[
        AIMessage(content='', tool_calls=[
            {'name': 'add', 'args': {'a': 3, 'b': 4}, 'id': 'c1', 'type': 'tool_call'}]),
        AIMessage(content='O resultado é 7.', tool_calls=[]),
    ])

# REGRA: um LLM (real ou Fake) por grafo. Reusar o MESMO Fake em vários grafos
# embaralha a sequência gravada (cada grafo consome falas); e criar Fake novo
# DENTRO do nó zera a sequência a cada volta (loop infinito, Aula 06).
def montar_fiscal(checkpointer=None, interrupt_before=None, interrupt_after=None,
                  thread='mesa-12'):
    llm = get_llm()  # fresco por grafo: sequência sempre começa do pedido
    def assistant(state: MessagesState) -> dict:
        return {'messages': [llm.invoke(
            [SystemMessage(content=SISTEMA)] + state['messages'])]}
    b = StateGraph(MessagesState)
    b.add_node('assistant', assistant)
    b.add_node('tools', ToolNode(TOOLS))
    b.add_edge(START, 'assistant')
    b.add_conditional_edges('assistant', tools_condition)
    b.add_edge('tools', 'assistant')
    g = b.compile(checkpointer=checkpointer,
                  interrupt_before=list(interrupt_before or []),
                  interrupt_after=list(interrupt_after or []))
    return g, {'configurable': {'thread_id': thread}}

fiscal, cfg = montar_fiscal(checkpointer=MemorySaver(), interrupt_before=['tools'])
print('agente com fiscal montado (pausa antes das tools).')

## 2. Conceito: pausar, inspecionar, retomar

- **Pausar**: `compile(..., interrupt_before=['tools'])` vale para **todo**
  `invoke`/`stream` desse grafo. Quando o fluxo chega no nó, ele **não executa**:
  emite `__interrupt__` e para, com o estado salvo no checkpoint.
- **Inspecionar**: `get_state(cfg)` devolve o retrato parado — `.next` diz para
  onde iria (`('tools',)`), `.values` tem o estado (inclui o `tool_calls`
  pendente para o humano ler).
- **Retomar**: `stream(None, cfg)` — `None` significa 'sem input novo, continua
  de onde parou'. O `invoke`/`stream` seguinte recomeça **após** o breakpoint.
- **Sem checkpointer não há retomada**: a pausa acontece (é no fluxo), mas
  `stream(None)` morre com `EmptyInputError` — não há sessão de onde continuar.

In [ ]:
# CÉLULA 3 — pausa + inspeção (rode e leia o pedido pendente)
for chunk in fiscal.stream({'messages': [HumanMessage(content='Some 3 e 4')]}, 
                           cfg, stream_mode='updates'):
    print(list(chunk.keys()))

estado = fiscal.get_state(cfg)
print('next:', estado.next)
pedido = estado.values['messages'][-1].tool_calls[0]
print('tool pendente:', pedido['name'], pedido['args'], '(id=' + pedido['id'] + ')')
assert estado.next == ('tools',)
print()
print('Leitura: o fiscal parou TUDO antes do caixa. Nada executou ainda.')

## 3. Dissecando a pausa (o que cada peça fez)

- `interrupt_before=['tools']` na compilação (não no nó, não no `invoke`).
  É configuração do **grafo**, não da chamada.
- O chunk `{'__interrupt__': ...}` **não é nó seu**: é o aviso do motor.
  Se seu log conta nós, ignore essa chave.
- `get_state(cfg).next == ('tools',)` — tupla (pode ter mais de um, em fan-out).
  `next == ()` significa 'parado no END, nada pendente'.
- `estado.values['messages'][-1]` é o `AIMessage` com `tool_calls` — exatamente
  o que o humano precisa ler para aprovar (nome + args + id).

In [ ]:
# CÉLULA 4 — aprovação humana + retomada (rode e carimbe)
try:
    decisao = input('Aprova add(3, 4)? [s]im / [n]ão (Enter = sim): ').strip().lower()
except Exception:
    decisao = 's'   # execução automática: aprova
    print('(sem terminal: aprovação automática)')

if decisao in ('', 's', 'sim', 'y', 'yes'):
    print('carimbado: retomando...')
    for chunk in fiscal.stream(None, cfg, stream_mode='updates'):
        print(list(chunk.keys()))
    final = fiscal.get_state(cfg)
    print('next:', final.next, '| resposta:', repr(final.values['messages'][-1].content))
    assert final.next == ()
    print('OK — tool executada APÓS o carimbo.')
else:
    print('rejeitado: o recibo nunca existiu. (Desviar a rota é a Aula 13.)')

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. Sem checkpointer: pausa sim, retomada não

A pausa é do **fluxo** (acontece de qualquer jeito); a retomada é da **sessão**
(precisa de onde continuar). Sem checkpointer, `stream(None)` morre com
`EmptyInputError: Received no input for __start__`.

In [ ]:
# CÉLULA 5 — pausa sem sessão (100% offline na lógica)
sem_sessao, _ = montar_fiscal(interrupt_before=['tools'])   # SEM checkpointer

for chunk in sem_sessao.stream({'messages': [HumanMessage(content='Some 3 e 4')]}):
    print(list(chunk.keys()))
print('pausou normalmente.')
try:
    list(sem_sessao.stream(None))
    print('retomou?! (não deveria)')
except Exception as erro:
    print(type(erro).__name__ + ':', str(erro)[:60])
    print('OK — pausar é do fluxo; retomar é da sessão.')

### 4b. `interrupt_after`: pausa DEPOIS do nó

`interrupt_before=['tools']` pausa antes do caixa; `interrupt_after=['assistant']`
pausa depois do atendente — aqui, param no mesmo ponto (`next == ('tools',)`).
A diferença aparece com nós que **escrevem antes de decidir**: `after` deixa o
efeito do nó gravado; `before` para antes de qualquer efeito.

In [ ]:
# CÉLULA 6 — interrupt_after (rode e compare o next)
depois, cfg2 = montar_fiscal(checkpointer=MemorySaver(), interrupt_after=['assistant'],
                              thread='mesa-13')
for chunk in depois.stream({'messages': [HumanMessage(content='Some 3 e 4')]}, 
                            cfg2, stream_mode='updates'):
    print(list(chunk.keys()))
print('next:', depois.get_state(cfg2).next)
assert depois.get_state(cfg2).next == ('tools',)
print('OK — mesmo ponto de parada, por outro motivo.')

## 5. Erros clássicos

1. **Retomar com input novo em vez de `None`.** `stream({'messages': [...]}, cfg)`
   após a pausa **recomeça** do zero na thread (anexa e reexecuta); `stream(None, cfg)`
   **continua** de onde parou. Para aprovar, é `None`.
2. **Esperar o breakpoint sem checkpointer e sem thread.** Pausar, pausa — mas
   sem sessão não há retomada (CÉLULA 5). Breakpoint pede os dois.
3. **Contar `__interrupt__` como nó.** É aviso do motor, não etapa sua. Filtros
   de log devem ignorá-lo.
4. **Aprovar sem ler o `tool_calls`.** Carimbar no escuro anula o fiscal: sempre
   inspecione `name` + `args` + `id` (CÉLULA 3) antes do `None`.
5. **Breakpoint em produção sem dono.** Pausa sem ninguém para retomar = pedido
   preso para sempre. Breakpoint exige fila/painel de aprovação com timeout.
6. **Achar que rejeitar é 'não retomar'.** Só não retomar deixa a thread parada
   (não cancelada). Desviar de verdade — editar ou pular o nó — é a Aula 13.

## 6. Exercício guiado — fiscal da conta acima de R$ 50

Situação: na lanchonete, conta acima de R$ 50 exige carimbo. Simule com o
fiscal pausando **toda** execução de tool: inspecione o pedido, confira que é
`add(30, 25)` = R$ 55 e retome. Valide que o recibo só existiu após o carimbo.

In [ ]:
# CÉLULA 7 — exercício guiado (fiscal da conta alta)
conta_alta, cfg_alta = montar_fiscal(checkpointer=MemorySaver(),
                                      interrupt_before=['tools'], thread='mesa-50')
if os.getenv('GROQ_API_KEY'):
    pergunta_alta = 'Some 30 e 25. Responda só o número.'
else:
    pergunta_alta = 'Some 3 e 4'   # Fake: sequência gravada

for chunk in conta_alta.stream({'messages': [HumanMessage(content=pergunta_alta)]}, 
                           cfg_alta, stream_mode='updates'):
    print(list(chunk.keys()))
estado_alta = conta_alta.get_state(cfg_alta)
pedido_alta = estado_alta.values['messages'][-1].tool_calls[0]
print('pedido parado no fiscal:', pedido_alta['name'], pedido_alta['args'])
assert estado_alta.next == ('tools',)
print('carimbando...')
for chunk in conta_alta.stream(None, cfg_alta, stream_mode='updates'):
    print(list(chunk.keys()))
final_alta = conta_alta.get_state(cfg_alta)
print('recibo:', repr(final_alta.values['messages'][-2].content))
assert final_alta.next == ()
print('OK — nada executou antes do carimbo.')

## 7. Desafio — fiscal que só pausa conta alta

Hoje o fiscal pausa **tudo** (`interrupt_before=['tools']` é estático). E se a
regra for 'pausar só conta acima de R$ 50'? Pense (sem rodar ainda):

- Com breakpoint estático, dá para decidir por **valor**? (Resposta: não — ele
  não lê `args`. É por isso que existe breakpoint **dinâmico**: a Aula 14.)
- Workaround com o que você já sabe: pausar sempre, mas no carimbo conferir
  `pedido['args']` e **só retomar** se aprovar (rejeitar = não retomar... com
  a ressalva do erro 6: a thread fica parada).
- Implemente o workaround para `add(30, 25)` (aprova, total 55) e para
  `add(2, 3)` (não retoma: mostre `next` ainda em `('tools',)`).

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] Onde o breakpoint mora: no nó, no `invoke` ou na compilação?
- [ ] O que é o chunk `__interrupt__` — e o que `get_state().next` diz?
- [ ] Por que retomar é `stream(None, cfg)` e não um novo `invoke` com input?
- [ ] Sem checkpointer: o que funciona (pausa) e o que morre (retomada)?
- [ ] Qual a diferença prática entre `interrupt_before` e `interrupt_after`?
- [ ] Na sua lanchonete, o que mereceria fiscal — e quem carimbaria?

**Próxima aula (13 — Feedback humano):** rejeitar de verdade — editar o estado
parado com `update_state`, corrigir o pedido e desviar a rota. O fiscal vira editor.